# Submit the Single-Step Merge Job

Load the workshop-local command-job YAML, replace its compute placeholder from `.env`, submit it, stream logs, and verify completion.

**Source:** Adapted from this repository's `notebooks/00_submit_azureml_pipelines.ipynb` and `pipelines/single-step-merge-job.yaml`.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import MLClient, load_job
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
RUN_JOB = os.getenv("RUN_SINGLE_STEP_JOB", "false").lower() in {"1", "true", "yes"}
pipeline_path = WORKSHOP_ROOT / "pipelines/single-step-merge-job.yaml"

In [ ]:
job = load_job(pipeline_path)
job.compute = f"azureml:{COMPUTE_NAME}"
job.display_name = "Workshop single-step taxi merge"
job.tags = {"workshop": "azureml-h2o", "operation": "single-step-merge"}

print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")

if RUN_JOB:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Job ended with status {final_job.status}")
    print(f"Merged output: {final_job.outputs['merged_data'].path}")
else:
    print("Submission disabled. Set RUN_SINGLE_STEP_JOB=true in workshop/.env.")

## Expected Result

The command job completes on the configured cluster and publishes a merged taxi CSV as its `merged_data` output.

Next: `02_submit_integration_compare.ipynb`.